# 21 — 본인 얼굴 4종 검증 + 마스크 Fine-tuning (v4)

> **목적 1:** 본인 4종 이미지 분류 검증 (live / print / replay / mask)  
> **목적 2:** 본인 칸예 마스크 사진 50장으로 mask 클래스 fine-tuning → Grad-CAM 경계면 개선  
>
> **실행 순서:**  
> Cell 0 → 1 → 2 (크롭 함수) → 3 (4종 업로드+검증) → 4 (시각화) → 5 (앱 복사)  
> → **Cell 6 (마스크 50장 업로드)** → 7 (fine-tuning) → 8 (재검증) → 9 (앱 재복사)

## Cell 0 — Google Drive 마운트

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
print('✅ Drive 마운트 완료')

## Cell 1 — 경로 설정 및 모델 로드

In [ ]:
import os, sys, json, shutil
import numpy as np
import cv2
import tensorflow as tf
from pathlib import Path

BASE       = '/content/drive/MyDrive/face-anti-spoofing'
SRC_DIR    = f'{BASE}/src'
TEST_DIR   = f'{BASE}/data/my_test_images'
DEMO_DIR   = f'{BASE}/data/demo_images'
REPORT_DIR = f'{BASE}/reports/phase6'
MODEL_PATH = f'{BASE}/models/stage2_webcam_v3.h5'
FT_MODEL_PATH = f'{BASE}/models/stage2_mask_ft.h5'  # fine-tuning 결과 저장
MASK_DIR   = f'{BASE}/data/my_mask_images'           # 마스크 50장 저장 폴더

os.makedirs(TEST_DIR,   exist_ok=True)
os.makedirs(DEMO_DIR,   exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(MASK_DIR,   exist_ok=True)
sys.path.insert(0, SRC_DIR)

from xai_explainer import explain

model = tf.keras.models.load_model(MODEL_PATH, compile=False)
print(f'✅ 모델 로드 완료')
print(f'   출력 헤드: {[o.name for o in model.outputs]}')
print(f'   GPU: {tf.config.list_physical_devices("GPU")}')

# LLaVA fallback
CAPTION_JSON = f'{BASE}/results/phase4/llava_captions.json'
llava_db = {}
if os.path.exists(CAPTION_JSON):
    with open(CAPTION_JSON, encoding='utf-8') as f:
        raw = json.load(f)
    if isinstance(raw, list):
        llava_db = {r.get('img_path', r.get('path','')): r.get('caption','')
                    for r in raw if isinstance(r, dict)}
    elif isinstance(raw, dict):
        llava_db = raw
    print(f'✅ LLaVA DB: {len(llava_db)}건')

FALLBACK_CAPTIONS = {
    'Live':   'Natural face — consistent skin texture, no artifacts.',
    'Print':  'Paper texture and reflection patterns — printed photo attack.',
    'Replay': 'Screen moire and pixel grid — replay attack.',
    'Mask':   'Rigid boundary and synthetic texture — 3D mask attack.',
}
LIVE_MEAN     = {'laplacian': 383.0, 'fft_high': 1134.0}
LAP_THRESHOLD = 157
SPOOF_KO = {
    0: 'Live (실제 얼굴)', 1: 'Print Attack (인쇄 공격)',
    2: 'Replay Attack (화면 재촬영)', 3: '3D Mask (입체 마스크)',
}
REGION_MAP = {
    'upper-center':'forehead region','upper-left':'forehead-left',
    'upper-right':'forehead-right','mid-center':'nose and cheek area',
    'mid-left':'left cheek','mid-right':'right cheek',
    'lower-center':'mouth and chin','lower-left':'lower-left jaw',
    'lower-right':'lower-right jaw','full-face':'entire face','none':'no region',
}
CATEGORIES = {
    'live'  : ('REAL', '본인 웹캠 Live'),
    'print' : ('FAKE', 'Print Attack'),
    'replay': ('FAKE', 'Replay Attack'),
    'mask'  : ('FAKE', 'Mask Attack'),
}

def get_caption(img_path_str, spoof_type_name):
    fname = Path(img_path_str).name if img_path_str else ''
    for k, v in llava_db.items():
        if fname and fname in k and v: return v
    for key in FALLBACK_CAPTIONS:
        if key.lower() in spoof_type_name.lower():
            return f'[Template] {FALLBACK_CAPTIONS[key]}'
    return f'[Template] {FALLBACK_CAPTIONS["Live"]}'

def detect_heatmap_region(heatmap_raw, threshold=0.5):
    if heatmap_raw is None: return 'unknown'
    hm = cv2.resize(heatmap_raw, (9,9))
    active = hm >= threshold
    if active.mean() > 0.6: return 'full-face'
    rows = np.where(active.any(axis=1))[0]
    cols = np.where(active.any(axis=0))[0]
    if len(rows) == 0: return 'none'
    v = 'upper' if rows.mean()<3 else ('lower' if rows.mean()>6 else 'mid')
    h = 'left'  if cols.mean()<3 else ('right' if cols.mean()>6 else 'center')
    return f'{v}-{h}'

def build_image_caption(verdict, spoof_type_idx, anchor_stats, heatmap_raw):
    lap = anchor_stats.get('laplacian', 0)
    fft = anchor_stats.get('fft_high', 0)
    region_desc = REGION_MAP.get(detect_heatmap_region(heatmap_raw), 'face region')
    lap_desc = (
        f'very low sharpness (Lap={lap:.0f})' if lap<LIVE_MEAN['laplacian']*0.5 else
        f'reduced sharpness (Lap={lap:.0f})'  if lap<LIVE_MEAN['laplacian']*0.8 else
        f'high edge contrast (Lap={lap:.0f})' if lap>LIVE_MEAN['laplacian']*1.2 else
        f'normal sharpness (Lap={lap:.0f})'
    )
    fft_desc = (
        f'low high-freq energy (FFT={fft:.0f})' if fft<LIVE_MEAN['fft_high']*0.6 else
        f'suppressed high-freq (FFT={fft:.0f})' if fft<LIVE_MEAN['fft_high']*0.85 else
        f'normal high-freq energy (FFT={fft:.0f})'
    )
    type_hints = {
        0: 'No spoofing artifacts — live face.',
        1: 'Paper-based attack: flat texture and ink dot pattern.',
        2: 'Screen replay: digital display interference pattern.',
        3: '3D mask: rigid boundary and synthetic texture inconsistency.',
    }
    return (f'Model focused on {region_desc}. '
            f'Texture: {lap_desc}, {fft_desc}. '
            f'{type_hints.get(spoof_type_idx, "")}')

def explain_v2(img_bgr, img_path_str=None, threshold=0.75, _model=None):
    _m = _model or model
    r  = explain(img_bgr, img_path=img_path_str, threshold=threshold)
    inp   = np.expand_dims(
        cv2.resize(cv2.cvtColor(img_bgr,cv2.COLOR_BGR2RGB),(224,224)).astype('float32')/255.0, 0
    )
    preds = _m.predict(inp, verbose=0)
    spoof_type_probs = ({i:float(p) for i,p in enumerate(preds[1][0])}
                        if isinstance(preds,list) and len(preds)>=2 else {})
    p_print = spoof_type_probs.get(1,0)
    p_mask  = spoof_type_probs.get(3,0)
    corrected = r['spoof_type_idx']
    if r['spoof_type_idx'] in (1,3) and abs(p_print-p_mask) < 0.3:
        corrected = 3 if r['anchor_stats'].get('laplacian',0) > LAP_THRESHOLD else 1
    r['spoof_type_idx']  = corrected
    r['spoof_type_name'] = SPOOF_KO.get(corrected, r['spoof_type_name'])
    if r['verdict'] == 'REAL':
        r['spoof_type_name'] = 'Live (실제 얼굴)'
    r['llava_caption'] = build_image_caption(
        r['verdict'], r['spoof_type_idx'], r['anchor_stats'], r['heatmap_raw']
    )
    return r

# 본인 4종 이미지 로드
my_images = {}
for cat in CATEGORIES:
    p = f'{TEST_DIR}/my_{cat}.jpg'
    if os.path.exists(p):
        img = cv2.imread(p)
        if img is not None:
            my_images[cat] = (img, p)
            print(f'  ✅ {cat}: {img.shape[1]}×{img.shape[0]}')
    else:
        print(f'  ⚠️ {cat}: 없음')

print(f'\n총 {len(my_images)}장 로드')

## Cell 2 — Haar Cascade 얼굴 크롭 함수

In [ ]:
face_cascade = cv2.CascadeClassifier(
    cv2.data.haarcascades + 'haarcascade_frontalface_default.xml'
)

def crop_face(img_bgr, target_size=224, margin_ratio=0.3):
    gray  = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    faces = face_cascade.detectMultiScale(gray, 1.1, 5, minSize=(60,60))
    if len(faces) == 0:
        faces = face_cascade.detectMultiScale(gray, 1.05, 3, minSize=(40,40))
    if len(faces) == 0:
        print('  ⚠️ 얼굴 감지 실패 → center crop')
        h,w = img_bgr.shape[:2]; s = min(h,w)
        return cv2.resize(img_bgr[(h-s)//2:(h+s)//2,(w-s)//2:(w+s)//2],(target_size,target_size))
    x,y,w,h = max(faces, key=lambda f:f[2]*f[3])
    ih,iw = img_bgr.shape[:2]
    mx,my = int(w*margin_ratio), int(h*margin_ratio)
    x1,y1 = max(0,x-mx), max(0,y-my)
    x2,y2 = min(iw,x+w+mx), min(ih,y+h+my)
    print(f'  얼굴 감지 ✅  ({x1},{y1})-({x2},{y2})')
    return cv2.resize(img_bgr[y1:y2,x1:x2],(target_size,target_size))

print('✅ crop_face() 준비 완료')

## Cell 3 — 본인 4종 업로드 + 검증

> 파일명에 `live` / `print` / `replay` / `mask` 키워드 포함 시 자동 인식

In [ ]:
from google.colab import files
from IPython.display import display, Image as IPImage

MARGIN_MAP = {'live':0.3,'print':0.3,'replay':0.3,'mask':0.05}

print('📤 4종 이미지 업로드 (live / print / replay / mask 키워드 포함)')
uploaded = files.upload()

for fname, fbytes in uploaded.items():
    arr = np.frombuffer(fbytes, dtype=np.uint8)
    img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if img is None: continue

    fname_lower = fname.lower()
    matched_cat = next((c for c in ['live','print','replay','mask'] if c in fname_lower), None)
    if matched_cat is None:
        print(f'⚠️ 인식 불가: {fname}'); continue

    print(f'\n[{matched_cat}] 원본: {img.shape[1]}×{img.shape[0]}')
    img_cropped = crop_face(img, margin_ratio=MARGIN_MAP[matched_cat])
    save_path   = f'{TEST_DIR}/my_{matched_cat}.jpg'
    cv2.imwrite(save_path, img_cropped)
    my_images[matched_cat] = (img_cropped, save_path)
    display(IPImage(save_path, width=160))

# 검증
print('\n' + '='*60)
print('  4종 분류 검증')
print('='*60)

test_results = {}
pass_count   = 0

for cat in ['live','print','replay','mask']:
    expected, label = CATEGORIES[cat]
    if cat not in my_images:
        print(f'\n[{cat}] ⚠️ 이미지 없음'); continue
    img_bgr, img_path = my_images[cat]
    r = explain_v2(img_bgr, img_path_str=img_path)
    test_results[cat] = r
    is_correct = (r['verdict'] == expected)
    if is_correct: pass_count += 1
    status = '✅ PASS' if is_correct else '❌ FAIL'
    print(f'\n[{cat.upper()}] {label}')
    print(f'  판정: {r["verdict"]} ({r["spoof_prob"]:.1%})  →  {status}')
    print(f'  유형: {r["spoof_type_name"]}')
    print(f'  Lap/FFT: {r["anchor_stats"]["laplacian"]:.1f} / {r["anchor_stats"]["fft_high"]:.1f}')

print(f'\n결과: {pass_count}/{len(test_results)} PASS')

## Cell 4 — XAI 시각화

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib

font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    matplotlib.rcParams['font.family'] = fm.FontProperties(fname=font_path).get_name()
matplotlib.rcParams['axes.unicode_minus'] = False

cat_order = [c for c in ['live','print','replay','mask'] if c in test_results]
n = len(cat_order)
fig = plt.figure(figsize=(18, 5*n))
fig.suptitle('본인 4종 XAI 검증', fontsize=15, fontweight='bold', y=1.01)

for row_i, cat in enumerate(cat_order):
    r = test_results[cat]
    img_bgr, _ = my_images[cat]
    expected, label = CATEGORIES[cat]
    is_correct = (r['verdict'] == expected)

    ax1 = fig.add_subplot(n,3,row_i*3+1)
    ax1.imshow(cv2.cvtColor(cv2.resize(img_bgr,(224,224)),cv2.COLOR_BGR2RGB))
    ax1.set_title(f'{label}\n{"✅ PASS" if is_correct else "❌ FAIL"}',
                  fontsize=10, fontweight='bold', color='black' if is_correct else 'red')
    ax1.axis('off')

    ax2 = fig.add_subplot(n,3,row_i*3+2)
    ax2.imshow(r['heatmap_overlay'])
    color = 'red' if r['verdict']=='FAKE' else 'green'
    region = detect_heatmap_region(r['heatmap_raw'])
    ax2.set_title(f'Grad-CAM\n{r["verdict"]} ({r["spoof_prob"]:.1%})\n활성: {region}',
                  fontsize=9, color=color)
    ax2.axis('off')

    ax3 = fig.add_subplot(n,3,row_i*3+3)
    ax3.axis('off')
    caption = r.get('llava_caption','') or '(없음)'
    summary = (
        f"[Layer 1] {r['verdict']} ({r['spoof_prob']:.1%})\n"
        f"[Layer 1] 유형: {r['spoof_type_name']}\n\n"
        f"[Layer 2] Lap: {r['anchor_stats']['laplacian']:.1f} / FFT: {r['anchor_stats']['fft_high']:.1f}\n"
        f"[Layer 2] {r['anchor_interp']}\n\n"
        f"[Layer 3] {caption[:130]}"
    )
    ax3.text(0.04, 0.97, summary, transform=ax3.transAxes, fontsize=8.5,
             verticalalignment='top',
             bbox=dict(boxstyle='round',
                       facecolor='lightgreen' if is_correct else 'lightyellow', alpha=0.85))
    ax3.set_title('Layer 2+3', fontsize=10)

plt.tight_layout()
out = f'{REPORT_DIR}/21_validation.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
plt.show()
print(f'✅ 저장: {out}')

## Cell 5 — 앱용 demo_images 복사

In [ ]:
APP_FILENAMES = {
    'live'  : ('01_webcam_live',   '01_webcam_live.jpg',   '본인 Live'),
    'print' : ('02_print_attack',  '02_print_attack.jpg',  'Print Attack'),
    'replay': ('03_replay_attack', '03_replay_attack.jpg', 'Replay Attack'),
    'mask'  : ('04_mask_attack',   '04_mask_attack.jpg',   'Mask Attack'),
}

demo_meta = {}
for cat, (app_key, app_fname, app_label) in APP_FILENAMES.items():
    src = f'{TEST_DIR}/my_{cat}.jpg'
    if not os.path.exists(src):
        print(f'  ⚠️ {cat}: 없음'); continue
    dst = f'{DEMO_DIR}/{app_fname}'
    shutil.copy(src, dst)
    r = test_results.get(cat)
    if r:
        demo_meta[app_key] = {
            'label': app_label, 'dst': dst,
            'verdict': r['verdict'], 'spoof_prob': float(r['spoof_prob']),
            'spoof_type_name': r['spoof_type_name'],
            'laplacian': int(r['anchor_stats']['laplacian']),
            'fft_high':  int(r['anchor_stats']['fft_high']),
            'llava_caption': r.get('llava_caption',''),
        }
    print(f'  ✅ {cat} → {app_fname}')

with open(f'{DEMO_DIR}/demo_meta.json','w',encoding='utf-8') as f:
    json.dump(demo_meta, f, ensure_ascii=False, indent=2)
print(f'\n✅ demo_meta.json 갱신: {len(demo_meta)}건')
print('→ 20번 노트북 Cell 3 실행하면 앱 시연 가능')

## ══════════════════════════════════════
## Phase 6-B: 마스크 Fine-tuning
## ══════════════════════════════════════

## Cell 6 — 마스크 50장 업로드 + 크롭

> **목적:** 본인 칸예 마스크 사진으로 mask 클래스 fine-tuning  
> → Grad-CAM이 마스크 경계면을 더 잘 포착하도록 개선  
>
> **촬영 가이드:**
> | 조건 | 권장 |
> |------|------|
> | 각도 | 정면 15장, 좌측 10장, 우측 10장, 위아래 각 5장, 멀리서 5장 |
> | 조명 | 밝은 곳 30장 + 어두운 곳 20장 |
> | 표정 | 상관없음 (마스크가 가리니까) |
>
> 파일명 규칙 불필요 — 전부 mask로 처리

In [ ]:
from google.colab import files
from IPython.display import display, Image as IPImage

print('📤 칸예 마스크 사진을 업로드하세요 (30~50장, 한 번에 선택)')
mask_uploaded = files.upload()

mask_images = []  # (img_cropped, save_path)

for i, (fname, fbytes) in enumerate(mask_uploaded.items()):
    arr = np.frombuffer(fbytes, dtype=np.uint8)
    img = cv2.imdecode(arr, cv2.IMREAD_COLOR)
    if img is None: continue

    # 마스크는 margin 작게 (경계선 포함)
    img_cropped = crop_face(img, margin_ratio=0.15)
    save_path   = f'{MASK_DIR}/mask_{i:03d}.jpg'
    cv2.imwrite(save_path, img_cropped)
    mask_images.append((img_cropped, save_path))

print(f'\n✅ 마스크 이미지 크롭 완료: {len(mask_images)}장')
print('  미리보기 (처음 5장):')
for _, p in mask_images[:5]:
    display(IPImage(p, width=120))

## Cell 7 — Mask Fine-tuning

> **전략:** spoof head만 추가 학습 (binary head는 frozen)  
> - 기존 CelebA mask 샘플 + 본인 마스크 이미지 혼합  
> - Epoch 10, LR 1e-4  
> - 결과 모델: `models/stage2_mask_ft.h5`

In [ ]:
from tensorflow.keras import optimizers
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import random

# ── 기존 CelebA mask 샘플 로드 ────────────────────────────────
CROP_DIR  = f'{BASE}/data/cropped'
celeba_mask_paths = sorted(Path(f'{CROP_DIR}/mask').glob('*.jpg'))[:100]
celeba_live_paths = sorted(Path(f'{CROP_DIR}/live').glob('*.jpg'))[:100]
print(f'CelebA mask: {len(celeba_mask_paths)}장')
print(f'CelebA live: {len(celeba_live_paths)}장')
print(f'본인 mask  : {len(mask_images)}장')

def load_img(path_or_arr, size=224):
    if isinstance(path_or_arr, (str, Path)):
        img = cv2.imread(str(path_or_arr))
    else:
        img = path_or_arr
    if img is None: return None
    img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    return cv2.resize(img_rgb, (size,size)).astype('float32') / 255.0

# ── 데이터셋 구성 ─────────────────────────────────────────────
# spoof_type: 0=live, 1=print, 2=replay, 3=mask
# binary:     0=live, 1=fake
X, y_bin, y_spoof = [], [], []

# CelebA live
for p in celeba_live_paths:
    arr = load_img(p)
    if arr is not None:
        X.append(arr); y_bin.append(0); y_spoof.append(0)

# CelebA mask
for p in celeba_mask_paths:
    arr = load_img(p)
    if arr is not None:
        X.append(arr); y_bin.append(1); y_spoof.append(3)

# 본인 마스크 (3배 oversampling)
for img_bgr, _ in mask_images * 3:
    arr = load_img(img_bgr)
    if arr is not None:
        X.append(arr); y_bin.append(1); y_spoof.append(3)

X       = np.array(X)
y_bin   = np.array(y_bin,   dtype='float32')
y_spoof = np.array(y_spoof, dtype='int32')

# 셔플
idx = np.random.permutation(len(X))
X, y_bin, y_spoof = X[idx], y_bin[idx], y_spoof[idx]

# train/val split
split = int(len(X) * 0.85)
X_tr, X_va = X[:split], X[split:]
y_bin_tr, y_bin_va   = y_bin[:split],   y_bin[split:]
y_sp_tr,  y_sp_va    = y_spoof[:split], y_spoof[split:]

print(f'\n학습 데이터: {len(X_tr)}장 | 검증: {len(X_va)}장')
print(f'mask 비율 (train): {(y_sp_tr==3).mean():.1%}')

# ── Fine-tuning 설정 ──────────────────────────────────────────
ft_model = tf.keras.models.load_model(MODEL_PATH, compile=False)

# binary head frozen, spoof head만 학습
for layer in ft_model.layers:
    if 'binary' in layer.name.lower():
        layer.trainable = False
    elif layer.name in ('spoof', 'shared', 'dense', 'dense_1'):
        layer.trainable = True
    else:
        layer.trainable = False

trainable = sum(1 for l in ft_model.layers if l.trainable)
print(f'학습 가능 레이어: {trainable}/{len(ft_model.layers)}')

ft_model.compile(
    optimizer=optimizers.Adam(learning_rate=1e-4),
    loss={
        'binary': 'binary_crossentropy',
        'spoof' : 'sparse_categorical_crossentropy',
    },
    loss_weights={'binary': 0.3, 'spoof': 0.7},  # spoof 학습에 집중
    metrics={'binary': ['accuracy'], 'spoof': ['accuracy']}
)

callbacks = [
    EarlyStopping(monitor='val_spoof_accuracy', patience=3,
                  restore_best_weights=True, mode='max'),
    ReduceLROnPlateau(monitor='val_spoof_accuracy', factor=0.5,
                     patience=2, mode='max'),
]

print('\n🚀 Fine-tuning 시작...')
history = ft_model.fit(
    X_tr,
    {'binary': y_bin_tr, 'spoof': y_sp_tr},
    validation_data=(X_va, {'binary': y_bin_va, 'spoof': y_sp_va}),
    epochs=15,
    batch_size=16,
    callbacks=callbacks,
    verbose=1,
)

ft_model.save(FT_MODEL_PATH)
print(f'\n✅ Fine-tuned 모델 저장: {FT_MODEL_PATH}')

## Cell 8 — Fine-tuning 결과 시각화 + 재검증

> 학습 곡선 + 4종 이미지로 before/after 비교

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
import matplotlib

font_path = '/usr/share/fonts/truetype/nanum/NanumGothic.ttf'
if os.path.exists(font_path):
    fm.fontManager.addfont(font_path)
    matplotlib.rcParams['font.family'] = fm.FontProperties(fname=font_path).get_name()
matplotlib.rcParams['axes.unicode_minus'] = False

# ── 학습 곡선 ────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(history.history['spoof_accuracy'],     label='train')
axes[0].plot(history.history['val_spoof_accuracy'], label='val')
axes[0].set_title('Spoof Type Accuracy'); axes[0].legend()
axes[1].plot(history.history['spoof_loss'],     label='train')
axes[1].plot(history.history['val_spoof_loss'], label='val')
axes[1].set_title('Spoof Type Loss'); axes[1].legend()
plt.tight_layout()
plt.savefig(f'{REPORT_DIR}/21_ft_curve.png', dpi=150)
plt.show()

# ── Before / After 비교 ───────────────────────────────────────
print('\n[Before vs After Fine-tuning — mask 집중 비교]\n')

if 'mask' in my_images:
    img_bgr, img_path = my_images['mask']

    r_before = explain_v2(img_bgr, img_path_str=img_path, _model=model)
    r_after  = explain_v2(img_bgr, img_path_str=img_path, _model=ft_model)

    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle('Mask Attack — Before vs After Fine-tuning', fontsize=13, fontweight='bold')

    for row, (label, r) in enumerate([('Before', r_before), ('After', r_after)]):
        img_rgb = cv2.cvtColor(cv2.resize(img_bgr,(224,224)), cv2.COLOR_BGR2RGB)
        axes[row][0].imshow(img_rgb)
        axes[row][0].set_title(f'{label}\n원본 (크롭됨)', fontsize=10)
        axes[row][0].axis('off')

        axes[row][1].imshow(r['heatmap_overlay'])
        region = detect_heatmap_region(r['heatmap_raw'])
        color  = 'red' if r['verdict']=='FAKE' else 'green'
        axes[row][1].set_title(
            f'Grad-CAM\n{r["verdict"]} ({r["spoof_prob"]:.1%})\n활성: {region}',
            fontsize=9, color=color)
        axes[row][1].axis('off')

        axes[row][2].axis('off')
        caption = r.get('llava_caption','')
        summary = (
            f"유형: {r['spoof_type_name']}\n"
            f"Lap: {r['anchor_stats']['laplacian']:.1f} / FFT: {r['anchor_stats']['fft_high']:.1f}\n\n"
            f"캡션:\n{caption[:150]}"
        )
        bg = 'lightgreen' if label=='After' else 'lightyellow'
        axes[row][2].text(0.04, 0.97, summary, transform=axes[row][2].transAxes,
                          fontsize=8.5, verticalalignment='top',
                          bbox=dict(boxstyle='round', facecolor=bg, alpha=0.85))
        axes[row][2].set_title(f'{label} XAI', fontsize=10)

    plt.tight_layout()
    out = f'{REPORT_DIR}/21_mask_before_after.png'
    plt.savefig(out, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'✅ 저장: {out}')

    print(f'\n히트맵 활성 영역')
    print(f'  Before: {detect_heatmap_region(r_before["heatmap_raw"])}')
    print(f'  After : {detect_heatmap_region(r_after["heatmap_raw"])}')

## Cell 9 — Fine-tuned 모델로 앱 업데이트

> After 히트맵이 마스크 경계면에 집중되면 이 셀 실행  
> `xai_explainer.py`의 MODEL_PATH를 fine-tuned 모델로 교체

In [ ]:
explainer_path = f'{SRC_DIR}/xai_explainer.py'
backup_path    = f'{SRC_DIR}/xai_explainer_before_ft.py'

# 백업
shutil.copy(explainer_path, backup_path)
print(f'✅ 백업: {backup_path}')

# MODEL_PATH 교체
with open(explainer_path, 'r', encoding='utf-8') as f:
    code = f.read()

new_code = code.replace(
    f'MODEL_PATH   = "{MODEL_PATH}"',
    f'MODEL_PATH   = "{FT_MODEL_PATH}"  # fine-tuned'
).replace(
    f'MODEL_PATH = "{MODEL_PATH}"',
    f'MODEL_PATH = "{FT_MODEL_PATH}"  # fine-tuned'
)

if new_code == code:
    print('⚠️ MODEL_PATH 자동 교체 실패 → 수동 확인 필요')
    print(f'   {FT_MODEL_PATH} 로 교체해주세요')
else:
    with open(explainer_path, 'w', encoding='utf-8') as f:
        f.write(new_code)
    print(f'✅ xai_explainer.py MODEL_PATH → fine-tuned 모델로 교체')

# demo_meta도 재업데이트
for cat, (app_key, app_fname, _) in APP_FILENAMES.items():
    if cat not in my_images: continue
    img_bgr, img_path = my_images[cat]
    r = explain_v2(img_bgr, img_path_str=img_path, _model=ft_model)
    if app_key in demo_meta:
        demo_meta[app_key].update({
            'verdict': r['verdict'],
            'spoof_prob': float(r['spoof_prob']),
            'spoof_type_name': r['spoof_type_name'],
            'llava_caption': r.get('llava_caption',''),
        })

with open(f'{DEMO_DIR}/demo_meta.json','w',encoding='utf-8') as f:
    json.dump(demo_meta, f, ensure_ascii=False, indent=2)

print('✅ demo_meta.json 갱신 완료')
print('→ 20번 노트북 Cell 3 재실행하면 fine-tuned 앱 시연 가능')

## ✅ 체크리스트

**기본 검증 (Cell 0~5)**
| 항목 | 확인 |
|------|------|
| 4종 이미지 4/4 PASS | ⬜ |
| Grad-CAM 히트맵 확인 | ⬜ |
| demo_meta.json 갱신 | ⬜ |

**마스크 Fine-tuning (Cell 6~9)**
| 항목 | 확인 |
|------|------|
| 마스크 30~50장 업로드 | ⬜ |
| Fine-tuning 완료 (val_spoof_acc 개선) | ⬜ |
| Before/After 히트맵 비교 — 경계면 집중 확인 | ⬜ |
| xai_explainer.py 모델 교체 | ⬜ |
| 20번 앱 재실행 | ⬜ |